In [1]:
# === SESSION BOOTSTRAP (run first in every notebook) ======================
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

PARENT = "/content/drive/MyDrive/UAV_TRUST_Research"
REPO   = f"{PARENT}/uav-trust-research"

for fname in (".gitconfig", ".git-credentials"):
    src = os.path.join(PARENT, fname)
    if os.path.exists(src):
        subprocess.run(f'cp "{src}" /root/{fname}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)

r = subprocess.run("git config --global user.name && git config --global user.email",
                   shell=True, capture_output=True, text=True)
print("git identity:", r.stdout.strip() or "MISSING - run 00_setup.ipynb first")

if os.path.isdir(REPO):
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    print("cwd:", os.getcwd())
else:
    print("Repository not on Drive yet - run 00_setup.ipynb first.")

Mounted at /content/drive
git identity: Md Anas Biswas
anasbiswas@gmail.com
cwd: /content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research


In [2]:
!pip install xgboost shap scikit-learn matplotlib pandas numpy scipy pyarrow requests --quiet

In [3]:
# Configuration: transfer-fragility mechanism on UAV-NIDD (same six families as notebook 04)
CONFIG = {
    "data_file": "data/uav_nidd/UAV-NDD CSV/UAV-Case1-Label.csv",
    "parquet_cache": "data/uav_nidd/case1.parquet",
    "encoding": "latin-1",
    "label_col": "Label",
    "normal_value": "Normal",
    "include_families": ["DDoS", "UDP Flooding", "MITM", "Jamming", "BruteForce", "De-authentication"],
    "subsample_n": 200000,
    "drop_col_patterns": [
        "unnamed", "index",
        "ip.src", "ip.dst", "srcport", "dstport", "udp.srcport", "udp.dstport",
        "frame.time", "frame.number", "time_epoch", "time_relative", "time_delta",
        "bssid", "mactime", "vendor_oui", "wlan_radio.timestamp", "wlan_radio.start_tsf",
        "radiotap.timestamp", "wlan.seq",
    ],
    "seeds": list(range(3)),         # 3 seeds for the mechanism pass (raise to match 04 later)
    "n_shap": 2000,
    "conformal_alpha": 0.10,
    "normal_fracs": {"train": 0.60, "cal": 0.20, "test_seen": 0.10, "test_shift": 0.10},
    "family_fracs": {"train": 0.60, "cal": 0.20, "test_seen": 0.20},
    "xgb": {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.1,
            "subsample": 0.9, "colsample_bytree": 0.9, "tree_method": "hist"},
    "fig_dir": "figures",
    "report_dir": "reports",
}
for d in [CONFIG["fig_dir"], CONFIG["report_dir"]]:
    os.makedirs(d, exist_ok=True)
CONFIG

{'data_file': 'data/uav_nidd/UAV-NDD CSV/UAV-Case1-Label.csv',
 'parquet_cache': 'data/uav_nidd/case1.parquet',
 'encoding': 'latin-1',
 'label_col': 'Label',
 'normal_value': 'Normal',
 'include_families': ['DDoS',
  'UDP Flooding',
  'MITM',
  'Jamming',
  'BruteForce',
  'De-authentication'],
 'subsample_n': 200000,
 'drop_col_patterns': ['unnamed',
  'index',
  'ip.src',
  'ip.dst',
  'srcport',
  'dstport',
  'udp.srcport',
  'udp.dstport',
  'frame.time',
  'frame.number',
  'time_epoch',
  'time_relative',
  'time_delta',
  'bssid',
  'mactime',
  'vendor_oui',
  'wlan_radio.timestamp',
  'wlan_radio.start_tsf',
  'radiotap.timestamp',
  'wlan.seq'],
 'seeds': [0, 1, 2],
 'n_shap': 2000,
 'conformal_alpha': 0.1,
 'normal_fracs': {'train': 0.6,
  'cal': 0.2,
  'test_seen': 0.1,
  'test_shift': 0.1},
 'family_fracs': {'train': 0.6, 'cal': 0.2, 'test_seen': 0.2},
 'xgb': {'n_estimators': 300,
  'max_depth': 6,
  'learning_rate': 0.1,
  'subsample': 0.9,
  'colsample_bytree': 0.9,
 

In [4]:
# Imports
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb, shap
from scipy.stats import spearmanr
from sklearn.metrics import balanced_accuracy_score
from src.trust import conformal_qhat, coverage_size
from src.data import prepare_splits
print("imports ready")

imports ready


In [5]:
# Load (parquet cache if present, else the latin-1 CSV; cache it for next time), then subsample
if os.path.exists(CONFIG["parquet_cache"]):
    df = pd.read_parquet(CONFIG["parquet_cache"])
else:
    df = pd.read_csv(CONFIG["data_file"], low_memory=False, encoding=CONFIG["encoding"])
    try: df.to_parquet(CONFIG["parquet_cache"]); print("cached to parquet")
    except Exception as e: print("parquet cache skipped:", e)
label_col, normal_value = CONFIG["label_col"], CONFIG["normal_value"]
if CONFIG["subsample_n"] and len(df) > CONFIG["subsample_n"]:
    frac = CONFIG["subsample_n"] / len(df)
    df = df.groupby(label_col, group_keys=False).sample(frac=frac, random_state=42).reset_index(drop=True)
keep = [normal_value] + list(CONFIG["include_families"])
df = df[df[label_col].isin(keep)].reset_index(drop=True)
families = list(CONFIG["include_families"])
print("shape:", df.shape, "| families:", families)

shape: (168881, 45) | families: ['DDoS', 'UDP Flooding', 'MITM', 'Jamming', 'BruteForce', 'De-authentication']


In [ ]:
# For each (family, seed): outcome signals (balanced accuracy, conformal coverage) and the
# transfer-fragility score (pro-attack attribution on seen attacks that reverses to pro-Normal
# on the unseen family). Store first-seed SHAP vectors for the attribution-transfer figure.
def sample(A, n, seed):
    if len(A) > n:
        j = np.random.default_rng(seed).choice(len(A), n, replace=False); return A[j]
    return A

def mean_shap(expl, Xg):
    s = np.asarray(expl.shap_values(Xg))
    if s.ndim == 3:
        s = s[..., 1]
    return s.mean(0)

rows, shap0 = [], {}
for seed in CONFIG["seeds"]:
    for F in families:
        S = prepare_splits(df, label_col, normal_value, F, CONFIG["drop_col_patterns"],
                           CONFIG["normal_fracs"], CONFIG["family_fracs"], seed)
        clf = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                random_state=seed, **CONFIG["xgb"]).fit(S["X_train"], S["y_train"])
        p_cal = clf.predict_proba(S["X_cal"])[:, 1]
        p_shift = clf.predict_proba(S["X_shift"])[:, 1]
        pr_h = (p_shift >= 0.5).astype(int)
        balacc = balanced_accuracy_score(S["y_shift"], pr_h)
        qhat = conformal_qhat(p_cal, S["y_cal"], alpha=CONFIG["conformal_alpha"])
        cov = coverage_size(p_shift, S["y_shift"], qhat)[0]

        Xa = sample(S["X_seen"][np.isin(S["fam_seen"], S["seen_families"])], CONFIG["n_shap"], seed)
        Xh = sample(S["X_shift"][S["fam_shift"] == F], CONFIG["n_shap"], seed)
        expl = shap.TreeExplainer(clf)
        m_seen = mean_shap(expl, Xa); m_held = mean_shap(expl, Xh)
        flip = np.maximum(m_seen, 0.0) * np.maximum(-m_held, 0.0)
        rows.append({"held_out": F, "seed": seed, "shift_balacc": balacc,
                     "shift_coverage": cov, "fragility": float(flip.sum())})
        if seed == CONFIG["seeds"][0]:
            shap0[F] = dict(feats=S["feature_names"], m_seen=m_seen, m_held=m_held,
                            flip=flip, imp=np.abs(m_seen))
    print("seed done:", seed)
raw = pd.DataFrame(rows)
print("runs:", raw.shape[0])

seed done: 0
seed done: 1


In [ ]:
# Per-family aggregate: fragility next to the two collapse signals (accuracy and coverage)
agg = raw.groupby("held_out").agg(
    fragility_mean=("fragility", "mean"), fragility_std=("fragility", "std"),
    balacc_mean=("shift_balacc", "mean"), balacc_std=("shift_balacc", "std"),
    coverage_mean=("shift_coverage", "mean"), coverage_std=("shift_coverage", "std")).sort_values("coverage_mean")
out = pd.DataFrame(index=agg.index)
out["fragility"] = agg.fragility_mean.round(3).astype(str) + " ± " + agg.fragility_std.round(3).astype(str)
out["shift_balacc"] = agg.balacc_mean.round(4).astype(str) + " ± " + agg.balacc_std.round(4).astype(str)
out["shift_coverage"] = agg.coverage_mean.round(4).astype(str) + " ± " + agg.coverage_std.round(4).astype(str)
print(out.to_string())
raw.to_csv(os.path.join(CONFIG["report_dir"], "07_uavnidd_fragility_raw.csv"), index=False)
agg.round(6).to_csv(os.path.join(CONFIG["report_dir"], "07_uavnidd_fragility_aggregate.csv"))

In [ ]:
# Which collapse signal does the fragility mechanism track: accuracy, coverage, or both?
r_acc, r_cov = [], []
for seed in CONFIG["seeds"]:
    d = raw[raw["seed"] == seed]
    r_acc.append(spearmanr(d["fragility"], d["shift_balacc"]).correlation)
    r_cov.append(spearmanr(d["fragility"], d["shift_coverage"]).correlation)
print("Spearman(fragility, balanced accuracy)  per seed:", [round(x, 2) for x in r_acc])
print("   mean %.3f  std %.3f" % (np.nanmean(r_acc), np.nanstd(r_acc)))
print("Spearman(fragility, conformal coverage) per seed:", [round(x, 2) for x in r_cov])
print("   mean %.3f  std %.3f" % (np.nanmean(r_cov), np.nanstd(r_cov)))
print("(negative => higher fragility predicts lower value of that signal)")

In [ ]:
# Hardest-reversing features for the family with the highest fragility (first seed)
Fmax = agg["fragility_mean"].idxmax()
d = shap0[Fmax]
print("Family with highest fragility:", Fmax)
for j in np.argsort(-d["flip"])[:10]:
    print("   %-26s seen=%+.3f  held-out=%+.3f  reversal=%.3f"
          % (d["feats"][j], d["m_seen"][j], d["m_held"][j], d["flip"][j]))

In [ ]:
# Figure A: fragility per family across seeds (is the ranking stable?)
fams = list(agg.index)
labels = [str(f)[:12] for f in fams]
jit = np.random.default_rng(0)
plt.figure(figsize=(9, 4.5))
for i, F in enumerate(fams):
    v = raw.loc[raw["held_out"] == F, "fragility"].values
    plt.scatter(i + jit.uniform(-0.12, 0.12, len(v)), v, s=24, alpha=0.6, color="#6a1b9a")
    plt.scatter([i], [v.mean()], marker="_", s=800, color="black", zorder=3)
plt.xticks(range(len(fams)), labels, rotation=20, ha="right")
plt.ylabel("transfer-fragility score")
plt.title("UAV-NIDD  |  transfer-fragility per family across %d seeds" % len(CONFIG["seeds"]))
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["fig_dir"], "07_uavnidd_fragility_spread.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Figure B: attribution transfer per family (first seed). Lower-right = transfer-fragile.
look = {r: (agg.loc[r, "fragility_mean"], agg.loc[r, "balacc_mean"], agg.loc[r, "coverage_mean"]) for r in agg.index}
ncol = 3; nrow = int(np.ceil(len(families) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4.3 * nrow))
axes = np.array(axes).flatten()
for a, F in zip(axes, families):
    d = shap0[F]
    lim = max(np.abs(d["m_seen"]).max(), np.abs(d["m_held"]).max()) * 1.15 + 1e-6
    a.axhline(0, color="gray", lw=.5); a.axvline(0, color="gray", lw=.5)
    a.plot([-lim, lim], [-lim, lim], "--", color="gray", lw=.8)
    a.scatter(d["m_seen"], d["m_held"], s=16, alpha=.5, color="#6a1b9a")
    for j in np.argsort(-d["imp"])[:4]:
        a.annotate(str(d["feats"][j]).replace("radiotap.", "").replace("wlan.", ""),
                   (d["m_seen"][j], d["m_held"][j]), fontsize=6)
    a.set_xlim(-lim, lim); a.set_ylim(-lim, lim)
    a.set_xlabel("mean SHAP, seen attacks"); a.set_ylabel("mean SHAP, held-out")
    fr, ba, cv = look[F]
    a.set_title("%s\nfrag=%.2f  acc=%.2f  cov=%.2f" % (str(F)[:14], fr, ba, cv), fontsize=9)
for a in axes[len(families):]:
    a.axis("off")
fig.suptitle("UAV-NIDD attribution transfer: lower-right quadrant = transfer-fragile features")
fig.tight_layout()
fig.savefig(os.path.join(CONFIG["fig_dir"], "07_uavnidd_attribution_transfer.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Commit results (end-of-unit discipline)
!git add reports/ figures/ notebooks/
!git commit -m "07 UAV-NIDD: transfer-fragility mechanism, correlated with both accuracy and coverage collapse"
!git push origin main